# 02 — Single-Point CIR Sanity Test

Test 1 (LOS with furniture): anchor_0 ceiling corner → room centre.
Test 2 (NLOS): anchor_3 floor → point behind bookshelf.

Assert Test 1: first-path range ≈ Euclidean distance to within 1 cm.
Assert Test 2: LOS flag False, first-path range > Euclidean distance.

In [ ]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
import sionna
import sionna.rt as rt
from sionna.rt import (PathSolver, Transmitter, Receiver,
                       PlanarArray, Camera, InteractionType)

print(f"Sionna {sionna.__version__}")

## Load scene (with furniture) and set frequency

In [ ]:
scene_path = os.path.abspath("../scene/room.xml")
scene = rt.load_scene(scene_path)
scene.frequency = 6.5e9          # DW3000 channel 5 centre frequency
print(f"Scene loaded  — frequency = {scene.frequency.item() / 1e9:.2f} GHz")
print(f"Wavelength    = {scene.wavelength.item() * 100:.2f} cm")
print(f"Objects in scene: {list(scene.objects.keys())}")

## Load anchor positions from config YAML

In [ ]:
anchors_yaml = os.path.abspath(
    "../../src/microuwb_bringup/config/anchors.yaml")
print(f"Reading: {anchors_yaml}")
with open(anchors_yaml, "r") as f:
    cfg = yaml.safe_load(f)

anchors = cfg["anchors"]
print("\nLoaded anchors:")
for a in anchors:
    print(f"  anchor_{a['id']}  ({a['x']}, {a['y']}, {a['z']})")

## Test 1: LOS — anchor_0 (ceiling) → room centre

The bookshelf, cafe table, and cabinet do not obstruct this path (verified geometrically: the line a0→centre passes above all furniture).

In [ ]:
a0 = anchors[0]
TX_POS = [float(a0["x"]), float(a0["y"]), float(a0["z"])]
RX_POS = [2.5, 2.0, 1.0]    # room centre at 1 m height

euclidean_dist = float(np.linalg.norm(np.array(TX_POS) - np.array(RX_POS)))

print(f"TX anchor_0 position : {TX_POS}")
print(f"RX room-centre       : {RX_POS}")
print(f"Euclidean distance   : {euclidean_dist:.6f} m")

## Configure antennas and add devices

In [ ]:
# Isotropic single-element arrays (DW3000 approximation)
iso_array = PlanarArray(num_rows=1, num_cols=1,
                        vertical_spacing=0.5, horizontal_spacing=0.5,
                        pattern="iso", polarization="V")
scene.tx_array = iso_array
scene.rx_array = iso_array

scene.add(Transmitter(name="tx", position=TX_POS))
scene.add(Receiver(name="rx",   position=RX_POS))
print("TX and RX added")

## Compute paths

In [ ]:
solver = PathSolver()
paths = solver(
    scene,
    max_depth=4,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False,
    diffraction=True,
    samples_per_src=int(1e6),
)
print(f"tau shape          : {paths.tau.shape}")
print(f"valid shape        : {paths.valid.shape}")
print(f"interactions shape : {paths.interactions.shape}")

## Extract first-path delay and compute range

In [ ]:
# tau shape with synthetic_array=True: (num_rx, num_tx, num_paths)
# valid: same shape
tau_all   = np.array(paths.tau).squeeze()     # → (num_paths,)
valid_all = np.array(paths.valid).squeeze()   # → (num_paths,)
valid_mask = valid_all.astype(bool)

tau_valid  = tau_all[valid_mask]
num_valid  = int(valid_mask.sum())

print(f"Total path slots : {len(tau_all)}")
print(f"Valid paths      : {num_valid}")

if num_valid == 0:
    raise RuntimeError(
        "No valid paths found.  Check that TX/RX are not inside walls "
        "and that scene normals point into the room.")

first_idx     = int(np.argmin(tau_valid))
first_delay_s = float(tau_valid[first_idx])

SPEED_OF_LIGHT = 3e8  # m/s
first_delay_ns  = first_delay_s * 1e9
first_range_m   = first_delay_s * SPEED_OF_LIGHT
range_error_cm  = abs(first_range_m - euclidean_dist) * 100

print()
print(f"First-path delay   : {first_delay_ns:.4f} ns")
print(f"First-path range   : {first_range_m:.4f} m")
print(f"Euclidean distance : {euclidean_dist:.4f} m")
print(f"Range error        : {range_error_cm:.3f} cm")

## LOS detection and Test 1 assertion

In [ ]:
inter = np.array(paths.interactions)
md_  = inter.shape[0]
np_  = inter.shape[-1]
inter_2d = inter.reshape(md_, -1, np_)[:, 0, :]      # (max_depth, num_paths)
inter_valid = inter_2d[:, valid_mask]
first_path_inter = inter_valid[:, first_idx]

los_flag_los = bool(np.all(first_path_inter == InteractionType.NONE))
err_los = abs(first_range_m - euclidean_dist)

print(f"First-path interactions : {first_path_inter.tolist()}")
print(f"LOS detected            : {los_flag_los}")
print(f"Range error             : {err_los * 100:.3f} cm")

TOLERANCE_M = 0.01   # 1 cm
test1_pass = los_flag_los and err_los < TOLERANCE_M

print()
if test1_pass:
    print("TEST 1 (LOS WITH FURNITURE) : PASSED")
    print(f"  LOS range error {err_los * 100:.3f} cm < {TOLERANCE_M * 100:.0f} cm tolerance")
else:
    msgs = []
    if not los_flag_los:
        msgs.append("First path NOT LOS — check normals or TX/RX placement.")
    if err_los >= TOLERANCE_M:
        msgs.append(f"Range error {err_los:.4f} m exceeds {TOLERANCE_M} m tolerance.")
    for m in msgs:
        print(f"FAIL: {m}")
    raise AssertionError("\n".join(msgs))

## Path visualisation (Test 1)

In [ ]:
import os
RENDERS = os.path.abspath("../renders")
os.makedirs(RENDERS, exist_ok=True)

cam = Camera(position=[-3.0, -2.0, 5.0], look_at=[2.5, 2.0, 1.5])
bmp = scene.render(camera=cam, paths=paths, num_samples=128,
                   resolution=(720, 540), show_devices=True,
                   return_bitmap=True)
img = np.array(bmp)
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img)
ax.set_title("Test 1: anchor_0 (ceiling) → room centre — LOS with furniture")
ax.axis("off")
plt.tight_layout()
out = os.path.join(RENDERS, "02_test1_los_paths.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

## Test 2: NLOS — anchor_3 (floor west) → behind bookshelf

anchor_3 is at (0.1, 2.0, 0.3) on the floor near the west wall.
The test RX at (0.5, 0.3, 0.5) is behind the bookshelf (bookshelf AABB: x=[0.05,0.95], y=[0.61,1.01], z=[0,1.2]).

Geometric verification: the line a3→RX enters the bookshelf at t≈0.58 (x=0.33, z=0.42) and exits at t≈0.82 (x=0.43, z=0.46).

Expected: LOS flag=False, first-path range > Euclidean distance.

In [ ]:
# Clear previous TX/RX before setting up NLOS test
scene.remove("tx")
scene.remove("rx")

a3 = anchors[3]
TX_NLOS = [float(a3["x"]), float(a3["y"]), float(a3["z"])]  # (0.1, 2.0, 0.3)
RX_NLOS = [0.5, 0.3, 0.5]   # south of bookshelf — geometrically blocked from a3

euclidean_nlos = float(np.linalg.norm(np.array(TX_NLOS) - np.array(RX_NLOS)))

print(f"TX anchor_3 (floor) : {TX_NLOS}")
print(f"RX (behind shelf)   : {RX_NLOS}")
print(f"Euclidean distance  : {euclidean_nlos:.4f} m")

scene.add(Transmitter(name="tx", position=TX_NLOS))
scene.add(Receiver(name="rx",   position=RX_NLOS))
print("NLOS TX/RX added")

In [ ]:
paths_nlos = solver(
    scene,
    max_depth=4,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False,
    diffraction=True,
    samples_per_src=int(1e6),
)
print(f"tau shape          : {paths_nlos.tau.shape}")
print(f"valid shape        : {paths_nlos.valid.shape}")

In [ ]:
tau_nlos   = np.array(paths_nlos.tau).squeeze()
valid_nlos = np.array(paths_nlos.valid).squeeze().astype(bool)
tau_v_nlos = tau_nlos[valid_nlos]
num_valid_nlos = int(valid_nlos.sum())

print(f"Valid paths (NLOS test) : {num_valid_nlos}")

if num_valid_nlos == 0:
    # No paths at all → extreme NLOS, test passes trivially
    los_flag_nlos = False
    first_range_nlos = float('inf')
    print("No valid paths — extreme NLOS (bookshelf fully blocking, no diffraction escape)")
else:
    first_idx_nlos   = int(np.argmin(tau_v_nlos))
    first_delay_nlos = float(tau_v_nlos[first_idx_nlos])
    first_range_nlos = first_delay_nlos * 3e8

    inter_nlos  = np.array(paths_nlos.interactions)
    md2  = inter_nlos.shape[0]
    np2  = inter_nlos.shape[-1]
    inter2d_nlos  = inter_nlos.reshape(md2, -1, np2)[:, 0, :]
    inter_v_nlos  = inter2d_nlos[:, valid_nlos]
    fp_inter_nlos = inter_v_nlos[:, first_idx_nlos]

    los_flag_nlos = bool(np.all(fp_inter_nlos == InteractionType.NONE))

    print(f"First-path interactions : {fp_inter_nlos.tolist()}")
    print(f"LOS detected            : {los_flag_nlos}")
    print(f"First-path range        : {first_range_nlos:.4f} m")
    print(f"Euclidean distance      : {euclidean_nlos:.4f} m")
    print(f"Range excess            : {(first_range_nlos - euclidean_nlos)*100:.1f} cm")

In [ ]:
# NLOS assertions: first path must NOT be LOS, and must be longer than Euclidean
test2_pass = (not los_flag_nlos) and (first_range_nlos > euclidean_nlos - 0.005)

print()
if test2_pass:
    print("TEST 2 (NLOS BEHIND BOOKSHELF) : PASSED")
    if num_valid_nlos > 0:
        excess_cm = (first_range_nlos - euclidean_nlos) * 100
        print(f"  LOS=False, range excess = {excess_cm:.1f} cm (NLOS path goes around bookshelf)")
    else:
        print("  LOS=False, no valid paths (bookshelf fully opaque at 6.5 GHz)")
else:
    msgs = []
    if los_flag_nlos:
        msgs.append(
            "First path classified as LOS — bookshelf may not be blocking the path.\n"
            "Check furniture_bookshelf.ply coordinates in gen_meshes.py.")
    if num_valid_nlos > 0 and first_range_nlos <= euclidean_nlos - 0.005:
        msgs.append(
            f"First-path range {first_range_nlos:.4f} m < Euclidean {euclidean_nlos:.4f} m "
            "— physically impossible for an NLOS path.")
    for m in msgs:
        print(f"FAIL: {m}")
    raise AssertionError("\n".join(msgs))

## NLOS path visualisation

In [ ]:
bmp_nlos = scene.render(camera=cam, paths=paths_nlos, num_samples=128,
                        resolution=(720, 540), show_devices=True,
                        return_bitmap=True)
img_nlos = np.array(bmp_nlos)
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img_nlos)
ax.set_title("Test 2: anchor_3 (floor) → behind bookshelf — NLOS")
ax.axis("off")
plt.tight_layout()
out = os.path.join(RENDERS, "02_test2_nlos_paths.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

## Summary

In [ ]:
print("=" * 55)
print("  CIR SANITY TEST SUMMARY")
print("=" * 55)
print(f"  Test 1 — LOS (anchor_0 ceiling → room centre)")
print(f"    TX:           {TX_POS}")
print(f"    RX:           {RX_POS}")
print(f"    Euclidean:    {euclidean_dist:.4f} m")
print(f"    Ray range:    {first_range_m:.4f} m")
print(f"    Error:        {abs(first_range_m - euclidean_dist)*100:.3f} cm")
print(f"    LOS flag:     {los_flag_los}")
print(f"    Result:       {'PASS' if test1_pass else 'FAIL'}")
print()
print(f"  Test 2 — NLOS (anchor_3 floor → behind bookshelf)")
print(f"    TX:           {TX_NLOS}")
print(f"    RX:           {RX_NLOS}")
print(f"    Euclidean:    {euclidean_nlos:.4f} m")
if num_valid_nlos > 0:
    print(f"    Ray range:    {first_range_nlos:.4f} m")
    print(f"    Range excess: {(first_range_nlos-euclidean_nlos)*100:.1f} cm")
else:
    print(f"    Ray range:    no valid paths")
print(f"    LOS flag:     {los_flag_nlos}")
print(f"    Result:       {'PASS' if test2_pass else 'FAIL'}")
print("=" * 55)

if test1_pass and test2_pass:
    print("  ALL TESTS PASSED — proceed to grid sweep (3a-ii)")
else:
    raise AssertionError("One or more CIR sanity tests failed.")